# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Melih-Yilmaz06/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

---

**Lane:** Refresh / Content Opportunity Scoring

| Item | Definition |
|---|---|
| **Grain** | One row = one URL's (content item's) search performance for a single `report_date`. In our analysis we aggregate to **one row per URL per month**. |
| **Table** | `fact_content_daily_performance` — GSC page-level daily performance, partitioned by `month`. Joined to `dim_content` for metadata. |
| **Time window** | Strictly `month='2026-03'` — a mid-panel month chosen to avoid the final sealed sample (June 2026). |
| **Label / proxy** | `needs_refresh`: 1 if a page's impressions dropped more than 20 % from the first half (days 1–15) to the second half (days 16–31) of the month. This directional proxy signals content staleness. |
| **Exclusion** | URLs with fewer than 10 total impressions in the month are excluded — ratios from tiny denominators produce noise, not signal. |

In [ ]:
!pip install -q duckdb pandas scikit-learn

import os, duckdb, pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN not found — set it as an env var or Colab secret.'

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

WH   = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"{WH}/fact_content_daily_performance"
DIM  = f"{WH}/dim_content"
MO   = '2026-03'

grain = con.execute(f"""
    SELECT report_date, client_id, content_id, COUNT(*) c
    FROM   '{FACT}/month={MO}/*.parquet'
    GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""").fetchdf()
print('✅ Grain holds — no duplicates.' if grain.empty else grain)

summary = con.execute(f"""
    SELECT COUNT(*) AS rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_id)  AS clients,
           COUNT(DISTINCT content_id) AS items
    FROM '{FACT}/month={MO}/*.parquet'
""").fetchdf()
print(summary.to_string(index=False))

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

---

| Bucket | Fields | Rationale |
|---|---|---|
| **Feature** | `past_30d_impressions`, `past_30d_clicks`, `avg_position`, `title_length`, `click_trend` | All are knowable at the decision moment — they rely strictly on historical search data or static metadata observed before the prediction window. |
| **Label** | `needs_refresh` (derived: 1 when second-half impressions drop > 20 % vs first-half) | The outcome we predict. Never used as a feature. |
| **Context** | `content_id`, `client_id`, `report_date`, `month` | Identifiers for grouping, joining, and splitting only. |
| **Excluded** | All `ga4_*` columns where `ga4_data_available IS NOT TRUE` (zeros = "not measured"); URLs with < 10 monthly impressions (unstable ratios); `second_half_impressions` (label-derived — see leakage demo). | GA4 zeros before a client's GA4 start date are misleading. Low-impression URLs add noise. |

In [ ]:
feature_df = con.execute(f"""
    WITH daily AS (
        SELECT content_id, client_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM   '{FACT}/month={MO}/*.parquet'
        WHERE  gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT content_id, client_id,
               SUM(gsc_impressions)  AS past_30d_impressions,
               SUM(gsc_clicks)       AS past_30d_clicks,
               AVG(gsc_avg_position) AS avg_position,
               SUM(CASE WHEN DAY(report_date)<=15 THEN gsc_impressions ELSE 0 END) AS h1_imp,
               SUM(CASE WHEN DAY(report_date)> 15 THEN gsc_impressions ELSE 0 END) AS h2_imp,
               SUM(CASE WHEN DAY(report_date)<=15 THEN gsc_clicks ELSE 0 END) AS h1_clk,
               SUM(CASE WHEN DAY(report_date)> 15 THEN gsc_clicks ELSE 0 END) AS h2_clk
        FROM   daily
        GROUP BY content_id, client_id
        HAVING SUM(gsc_impressions) >= 10   -- exclusion rule
    )
    SELECT a.*,
           CASE WHEN h1_clk>0 THEN (h2_clk-h1_clk)*1.0/h1_clk ELSE 0 END AS click_trend,
           LENGTH(COALESCE(d.title,''))                                     AS title_length,
           CASE WHEN h1_imp>0 AND (h2_imp-h1_imp)*1.0/h1_imp < -0.20
                THEN 1 ELSE 0 END                                           AS needs_refresh
    FROM agg a LEFT JOIN '{DIM}/*.parquet' d ON a.content_id = d.content_id
""").fetchdf()

FEATURES = ['past_30d_clicks','past_30d_impressions','avg_position','title_length','click_trend']
LABEL    = 'needs_refresh'

df = feature_df[FEATURES + [LABEL, 'h2_imp']].fillna(0)
print(f'Feature frame: {df.shape[0]} rows × {len(FEATURES)} features')
print(f'Label balance: {df[LABEL].value_counts().to_dict()}')

LEAKED = FEATURES + ['h2_imp']
scores_leak = cross_val_score(
    DecisionTreeClassifier(max_depth=5, random_state=42),
    df[LEAKED], df[LABEL], cv=5, scoring='roc_auc')
print(f'\n🚨 WITH leakage (h2_imp):   AUC = {scores_leak.mean():.4f} ± {scores_leak.std():.4f}')

scores_ok = cross_val_score(
    DecisionTreeClassifier(max_depth=5, random_state=42),
    df[FEATURES], df[LABEL], cv=5, scoring='roc_auc')
print(f'✅ WITHOUT leakage:         AUC = {scores_ok.mean():.4f} ± {scores_ok.std():.4f}')
print(f'\n→ Leakage inflates the score to ~1.0 because h2_imp directly encodes the label.')
print(f'  After dropping it the model must earn its score from honest historical features.')

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

---

Three proving queries follow:

1. **Grain** — prove `(report_date, client_id, content_id)` is unique (zero duplicates).
2. **Row count & date span** — total rows, MIN/MAX dates for the `2026-03` partition.
3. **Availability** — how many rows survive `gsc_data_available IS TRUE` and `gsc_impressions IS NOT NULL`.

In [ ]:
q1 = con.execute(f"""
    SELECT report_date, client_id, content_id, COUNT(*) c
    FROM   '{FACT}/month={MO}/*.parquet'
    GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""").fetchdf()
print('Q1 – Grain:', '✅ unique' if q1.empty else q1)

q2 = con.execute(f"""
    SELECT COUNT(*) AS total_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM '{FACT}/month={MO}/*.parquet'
""").fetchdf()
print('\nQ2 – Row count & date span:')
print(q2.to_string(index=False))

q3 = con.execute(f"""
    SELECT COUNT(*) AS total,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)          AS gsc_ok,
           COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL AND gsc_impressions >= 1) AS has_imp
    FROM '{FACT}/month={MO}/*.parquet'
""").fetchdf()
print('\nQ3 – Availability:')
print(q3.to_string(index=False))
t = q3['total'].iloc[0]; s = q3['gsc_ok'].iloc[0]
print(f'→ {s:,}/{t:,} ({s/t*100:.1f}%) rows pass gsc_data_available IS TRUE')

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

---

1. **No competitor context.** This slice only observes our own site's impression drops but lacks competitor content data to understand *why* content became outdated. A page might lose impressions because a competitor published superior content — we cannot observe that.

2. **Unbalanced panel depth.** Per-client history length varies wildly (`dim_clients.gsc_data_start`). A single-month slice cannot distinguish seasonal dips from structural content decay.

3. **GSC-only signal.** We deliberately filtered out rows where `ga4_data_available IS NOT TRUE` to avoid treating zero-filled GA4 columns as real engagement. This means we have no user-behaviour features (scroll depth, time on page) for our predictions.

4. **Within-month trend is noisy.** Splitting one month into two 15-day halves is a coarse signal — day-of-week effects, holidays, or a single viral day can dominate. A multi-month lookback would be more robust.

5. **Survivorship bias.** By excluding URLs with < 10 impressions we remove the long tail of pages that may be the *most* in need of refreshing — they have already dropped off the radar entirely.

In [ ]:
limits = con.execute(f"""
    SELECT client_id,
           COUNT(DISTINCT report_date) AS days_observed,
           MIN(report_date)            AS first_date,
           MAX(report_date)            AS last_date
    FROM '{FACT}/month={MO}/*.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_id
    ORDER BY days_observed
    LIMIT 10
""").fetchdf()
print('Clients with fewest observed days in 2026-03:')
print(limits.to_string(index=False))
print('\n→ Panel depth is uneven — even within a single month some clients have gaps.')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.